In [ ]:
# imports

import os
import re
import random
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T

In [ ]:
# put the main settings in one place
# checkpoint_source chooses which trained resnet18 backbone to load
# this notebook does not include a randomized resnet condition

config = {
    # cidentification, individuation
    "checkpoint_source": "identification",

    # data paths
    "pairs_dir": "/zpool/vladlab/data_drive/geogaze_data/decoder_stimulus/pairs",
    "left_masks_dir": "/zpool/vladlab/data_drive/geogaze_data/decoder_stimulus/left_masks",
    "right_masks_dir": "/zpool/vladlab/data_drive/geogaze_data/decoder_stimulus/right_masks",

    # trained resnet18 checkpoints
    # these should be the checkpoints from the resnet18 identification/individuation training notebooks
    "checkpoint_paths": {
        "identification": "/zpool/vladlab/data_drive/geogaze_data/cornet_coco_bboxes/resnet18/identification_critical/best.pth.tar",
        "individuation": "/zpool/vladlab/data_drive/geogaze_data/cornet_coco_bboxes/resnet18/individuation_critical/best.pth.tar",
    },

    # output root for the geogaze models
    "output_root": "/zpool/vladlab/data_drive/geogaze_data/geogaze_final/resnet18_geogaze_task",

    # test image folders
    # these match the original resnet geogaze notebooks
    "test_image_roots": {
        "identification": "/zpool/vladlab/data_drive/geogaze_data/decoder_stimulus/test_stimuli/identification_resnet/test_images",
        "individuation": "/zpool/vladlab/data_drive/geogaze_data/decoder_stimulus/test_stimuli/individuation_resnet/test_images",
    },

    # run all pair conditions
    "run_conditions": [
        ("L", "bc_bs"),
        ("L", "bc_gc"),
        ("L", "bs_bc"),
        ("L", "bs_gs"),
        ("L", "gc_bc"),
        ("L", "gc_gs"),
        ("L", "gs_bs"),
        ("L", "gs_gc"),
        ("R", "bc_bs"),
        ("R", "bc_gc"),
        ("R", "bs_bc"),
        ("R", "bs_gs"),
        ("R", "gc_bc"),
        ("R", "gc_gs"),
        ("R", "gs_bs"),
        ("R", "gs_gc"),
    ],

    # training settings
    "seed": 1,
    "epochs": 200,
    "batch_size": 32,
    "workers": 8,
    "val_split": 0.2,
    "lr": 0.01,
    "momentum": 0.9,
    "weight_decay": 1e-4,
    "threshold": 0.2,

    # image settings
    "img_size": 224,
}

assert config["checkpoint_source"] in ["identification", "individuation"]

pairs_dir = Path(config["pairs_dir"])
checkpoint_path = Path(config["checkpoint_paths"][config["checkpoint_source"]])
test_img_root = Path(config["test_image_roots"][config["checkpoint_source"]])

output_root = Path(config["output_root"]) / f"{config['checkpoint_source']}_geogaze_task"
output_root.mkdir(parents=True, exist_ok=True)

print("checkpoint source:", config["checkpoint_source"])
print("checkpoint path:", checkpoint_path)
print("output root:", output_root)
print("test image root:", test_img_root)
print("number of geogaze runs:", len(config["run_conditions"]))

In [ ]:
# set seed and device
# this makes the train/val split repeatable

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("torch:", torch.__version__)

In [ ]:
# image transforms
# resnet expects 224 x 224 images with imagenet normalization

img_tf = T.Compose([
    T.Resize((config["img_size"], config["img_size"])),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
def load_mask_binary(path):
    # object pixels are black in the masks
    mask = Image.open(path).convert("L")
    mask = mask.resize((config["img_size"], config["img_size"]), resample=Image.NEAREST)
    mask = np.asarray(mask, dtype=np.uint8)
    mask = (mask < 128).astype(np.float32)

    return torch.from_numpy(mask).unsqueeze(0)

In [ ]:
# dataset for image-mask pairs
# each item returns the pair image, the binary mask, and an id string
class PairMaskDataset(Dataset):
    def __init__(self, items, img_transform):
        self.items = items
        self.img_transform = img_transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, mask_path, mid, id_ = self.items[idx]

        image = Image.open(img_path).convert("RGB")
        image = self.img_transform(image)

        mask = load_mask_binary(mask_path)

        return image, mask, f"{mid}_{id_}"

In [ ]:
# dataset for image-mask pairs
# each item returns the pair image, the binary mask, and an id string
class PairMaskDataset(Dataset):
    def __init__(self, items, img_transform):
        self.items = items
        self.img_transform = img_transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, mask_path, mid, id_ = self.items[idx]

        image = Image.open(img_path).convert("RGB")
        image = self.img_transform(image)

        mask = load_mask_binary(mask_path)

        return image, mask, f"{mid}_{id_}"

In [ ]:
# collect image-mask pairs for one mask side and one pair condition

def get_masks_dir(mask_side):
    if mask_side == "L":
        return Path(config["left_masks_dir"])

    if mask_side == "R":
        return Path(config["right_masks_dir"])

    raise ValueError("mask_side must be L or R")


def collect_items(mask_side, pair_mid):
    masks_dir = get_masks_dir(mask_side)
    pair_re = re.compile(rf"^pair_{re.escape(pair_mid)}_(\d+)\.png$")

    items = []
    skipped = 0

    for fn in os.listdir(pairs_dir):
        match = pair_re.match(fn)
        if not match:
            continue

        id_ = match.group(1)
        img_path = pairs_dir / fn
        mask_path = masks_dir / f"mask{mask_side}_{pair_mid}_{id_}.png"

        if not mask_path.is_file():
            skipped += 1
            continue

        items.append((img_path, mask_path, pair_mid, id_))

    items.sort(key=lambda x: int(x[3]))

    if skipped:
        print(f"warning: skipped {skipped} image(s) with no matching mask{mask_side}")

    return items

In [ ]:
# make train and validation loaders for one condition
def make_loaders(items):
    rng = random.Random(config["seed"])
    shuffled = items[:]
    rng.shuffle(shuffled)
    n_val = max(1, int(len(shuffled) * config["val_split"]))
    val_items = shuffled[:n_val]
    train_items = shuffled[n_val:]
    train_loader = DataLoader(
        PairMaskDataset(train_items, img_tf),
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=config["workers"],
        pin_memory=(device.type == "cuda"),
    )

    val_loader = DataLoader(
        PairMaskDataset(val_items, img_tf),
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["workers"],
        pin_memory=(device.type == "cuda"),
    )

    return train_loader, val_loader, train_items, val_items

In [ ]:
# resnet18 feature extractor
# this uses the same backbone structure as the resnet18 training notebooks
# it returns the last spatial feature map before the classifier

class ResNet18SpatialBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = timm.create_model(
            "resnet18",
            pretrained=False,
            features_only=True,
        )

    def forward(self, x):
        feats = self.model(x)
        # use the last feature map
        return feats[-1]

In [ ]:
# load only the trained resnet18 backbone weights from a checkpoint
# the old identification/individuation heads are ignored

def load_resnet_backbone_weights(backbone, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt["model"] if "model" in ckpt else ckpt

    backbone_state = {}
    for key, value in state.items():
        if key.startswith("backbone.model."):
            backbone_state[key.replace("backbone.model.", "")] = value
        elif key.startswith("module.backbone.model."):
            backbone_state[key.replace("module.backbone.model.", "")] = value
        elif key.startswith("encoder.model."):
            backbone_state[key.replace("encoder.model.", "")] = value

    if len(backbone_state) == 0:
        raise RuntimeError(f"no resnet18 backbone weights found in {ckpt_path}")

    missing, unexpected = backbone.model.load_state_dict(backbone_state, strict=False)

    print("loaded resnet18 backbone from:", ckpt_path)
    print("missing keys:", len(missing))
    print("unexpected keys:", len(unexpected))

    return backbone

In [ ]:
# build the frozen encoder once
# this is the only part that changes when you switch checkpoint_source
encoder = ResNet18SpatialBackbone().to(device)
encoder = load_resnet_backbone_weights(encoder, checkpoint_path)
for p in encoder.parameters():
    p.requires_grad = False

encoder.eval()

print("encoder source:", config["checkpoint_source"])
print(encoder)

In [ ]:
# mask head
# takes the resnet18 feature map and predicts a one-channel object mask
class MaskHead(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, 1, kernel_size=1)
    def forward(self, feats):
        logits = self.proj(feats)
        logits = F.interpolate(
            logits,
            size=(config["img_size"], config["img_size"]),
            mode="bilinear",
            align_corners=False,
        )

        return logits
class ResNetMaskModel(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()

        self.encoder = encoder
        self.head = head
    def forward(self, x):
        feats = self.encoder(x)
        logits = self.head(feats)

        return logits

In [ ]:
# mask head
# takes the resnet18 feature map and predicts a one-channel object mask

class MaskHead(nn.Module):
    def __init__(self, in_ch):
        super().__init__()

        self.proj = nn.Conv2d(in_ch, 1, kernel_size=1)
    def forward(self, feats):
        logits = self.proj(feats)
        logits = F.interpolate(
            logits,
            size=(config["img_size"], config["img_size"]),
            mode="bilinear",
            align_corners=False,
        )

        return logits
class ResNetMaskModel(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()

        self.encoder = encoder
        self.head = head

    def forward(self, x):
        feats = self.encoder(x)
        logits = self.head(feats)

        return logits

In [ ]:
# find how many resnet channels the mask head needs

with torch.no_grad():
    dummy = torch.zeros(1, 3, config["img_size"], config["img_size"], device=device)
    feats = encoder(dummy)
    in_ch = feats.shape[1]
print("encoder feature shape:", feats.shape)
print("mask head input channels:", in_ch)

In [ ]:
# build a fresh mask model for one condition
# each condition gets a new head, but the frozen encoder source stays the same
def build_mask_model():
    head = MaskHead(in_ch).to(device)
    model = ResNetMaskModel(encoder, head).to(device)

    for p in model.encoder.parameters():
        p.requires_grad = False

    for p in model.head.parameters():
        p.requires_grad = True

    return model

In [ ]:
# train one geogaze head
def train_one_condition(model, train_loader, val_loader, run_dir, run_config):
    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.SGD(
        model.head.parameters(),
        lr=config["lr"],
        momentum=config["momentum"],
        weight_decay=config["weight_decay"],
    )

    best_val_loss = float("inf")
    history = []

    for epoch in range(config["epochs"]):
        model.train()
        model.encoder.eval()
        train_loss = 0.0
        for images, masks, ids in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            loss = criterion(logits, masks)

            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for images, masks, ids in val_loader:
                images = images.to(device)
                masks = masks.to(device)

                logits = model(images)
                loss = criterion(logits, masks)

                val_loss += loss.item()

        val_loss /= len(val_loader)

        history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss})

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.head.state_dict(), run_dir / "mask_head_best.pt")
            torch.save(
                {
                    "model": model.state_dict(),
                    "config": run_config,
                    "epoch": epoch,
                    "best_val_loss": best_val_loss,
                },
                run_dir / "best_full_model.pt",
            )

        torch.save(model.head.state_dict(), run_dir / "mask_head_latest.pt")

        print(
            f"epoch {epoch + 1}/{config['epochs']} | "
            f"train loss: {train_loss:.4f} | "
            f"val loss: {val_loss:.4f}"
        )
    pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)

    return best_val_loss

In [ ]:
# use the original resnet geogaze tag style
# this keeps the expected test-image folders

def make_model_tag(mask_side, pair_mid):
    if config["checkpoint_source"] == "identification":
        prefix = "resnetIDEN"
    elif config["checkpoint_source"] == "individuation":
        prefix = "resnetIDIV"
    else:
        raise ValueError("checkpoint_source must be identification or individuation")
    return f"{prefix}_mask{mask_side}_{pair_mid}"

In [ ]:
# save probability maps for the test images
# this follows the old notebooks: find test images in test_img_root / model_tag
# then save each probability map csv inside the run_dir

def save_test_probability_maps(model, model_tag, run_dir):
    pred_map_folder = test_img_root / model_tag
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp")
    if not pred_map_folder.exists():
        print(f"warning: no test image folder found: {pred_map_folder}")
        return 0

    image_paths = sorted([p for p in pred_map_folder.iterdir() if p.is_file() and p.suffix.lower() in exts])

    print(f"found {len(image_paths)} test images in: {pred_map_folder}")
    model.eval()
    with torch.no_grad():
        for img_path in image_paths:
            image = Image.open(img_path).convert("RGB")
            x = img_tf(image).unsqueeze(0).to(device)

            logits = model(x)
            probs = torch.sigmoid(logits)
            prob_map = probs.squeeze().detach().cpu().numpy()
            out_name = f"{img_path.stem}_prob_map.csv"
            out_path = run_dir / out_name
            pd.DataFrame(prob_map).to_csv(out_path, index=False)
            print(f"saved: {out_path}")

    return len(image_paths)

In [ ]:
# run all mask-side / pair conditions
# each condition gets itsh separated outputs
summary_rows = []

for run_idx, (mask_side, pair_mid) in enumerate(config["run_conditions"], start=1):
    model_tag = make_model_tag(mask_side, pair_mid)
    run_dir = output_root / model_tag
    run_dir.mkdir(parents=True, exist_ok=True)
    run_config = dict(config)
    run_config["mask_side"] = mask_side
    run_config["pair_mid"] = pair_mid
    run_config["model_tag"] = model_tag
    run_config["run_dir"] = str(run_dir)
    run_config["checkpoint_path"] = str(checkpoint_path)

    with open(run_dir / "config.json", "w") as f:
        json.dump(run_config, f, indent=2)
    print("\n" + "=" * 100)
    print(f"run {run_idx}/{len(config['run_conditions'])}: {model_tag}")

    items = collect_items(mask_side, pair_mid)
    if len(items) == 0:
        print(f"warning: no training items found for {model_tag}; skipping")
        summary_rows.append({
            "model_tag": model_tag,
            "mask_side": mask_side,
            "pair_mid": pair_mid,
            "n_train_items": 0,
            "n_val_items": 0,
            "best_val_loss": np.nan,
            "n_test_csvs": 0,
            "status": "skipped_no_training_items",
        })
        continue
    train_loader, val_loader, train_items, val_items = make_loaders(items)

    print("train:", len(train_items))
    print("val:", len(val_items))

    model = build_mask_model()
    best_val_loss = train_one_condition(model, train_loader, val_loader, run_dir, run_config)

    # reload the best head before making the test csvs
    best_head_path = run_dir / "mask_head_best.pt"
    model.head.load_state_dict(torch.load(best_head_path, map_location=device))

    n_test_csvs = save_test_probability_maps(model, model_tag, run_dir)

    summary_rows.append({
        "model_tag": model_tag,
        "mask_side": mask_side,
        "pair_mid": pair_mid,
        "n_train_items": len(train_items),
        "n_val_items": len(val_items),
        "best_val_loss": best_val_loss,
        "n_test_csvs": n_test_csvs,
        "status": "done",
    })
summary = pd.DataFrame(summary_rows)
summary_path = output_root / f"summary_{config['checkpoint_source']}.csv"
summary.to_csv(summary_path, index=False)

print("\nall runs done")
print("summary saved to:", summary_path)
summary